# MediSense-Cardio -- own-phone photos of ECG pages on a screen (Kaggle, GPU)

**Why:** to confirm on a NEW phone whether the five photo scores predict that the digitiser gives no output. Rules were fixed before any photo existed (docs/plan/analysis_plan_changes.md, 2026-11-08).
**What it does:** reads your 60 photos, digitises each one with the same digitiser and settings as the main study, computes the five photo scores, and saves the results. Matching the photos to the pages
and all the statistics are done afterwards on your computer.

**Steps (Kaggle):**
1. Put your 60 photos in one folder on your computer. Kaggle -> Create -> New Dataset -> upload the folder -> name it `ownphotos` -> Create (keep it Private).
2. New notebook -> File -> Import notebook -> upload this file.
3. Right panel -> Input -> Add Input -> your dataset `ownphotos`.
4. Settings: Accelerator = GPU T4 x2 (or P100), Internet = On.
5. Save Version -> Save & Run All (Commit). About 30-40 minutes. When the Logs show `ALL OWN PHOTOS COMPLETE`, open Output and download the zip. Tell me when it is in Downloads.


## Step 1 -- Find the photos


In [ ]:
import os, glob, json, time, shutil, subprocess, sys, datetime
import numpy as np, pandas as pd

WORK = '/kaggle/working'            # what is saved and can be downloaded
TMP = '/tmp/ownphotos'              # scratch space, not saved
INPUT_ROOT = '/kaggle/input'
DIG_DIR = f'{TMP}/Open-ECG-Digitizer'
os.makedirs(WORK, exist_ok=True); os.makedirs(TMP, exist_ok=True)

exts = ('.jpg', '.jpeg', '.png')
allfiles = [p for p in glob.glob(INPUT_ROOT + '/**/*', recursive=True) if os.path.isfile(p)]
photos = sorted(p for p in allfiles if p.lower().endswith(exts))
others = sorted(set(os.path.splitext(p)[1].lower() for p in allfiles if not p.lower().endswith(exts)))
print(f'{len(photos)} photo files found. Other file types present: {others}')
if any(e in others for e in ('.heic', '.heif')):
    raise SystemExit('STOPPED on purpose: some photos are HEIC (iPhone default). Set the iPhone camera to Most Compatible (JPEG) or convert them to JPG, upload again, and Run All.')
assert len(photos) > 0, 'No photos found. Right panel -> Input -> Add Input -> choose your dataset ownphotos. Then Run All again.'
names = [os.path.basename(p) for p in photos]
assert len(set(names)) == len(names), 'Two photos have the same file name: rename them so every name is different.'
print('first five:', names[:5])
if len(photos) != 60:
    print(f'NOTE: expected 60 photos, found {len(photos)}. The notebook will still run; tell me the number.')
all_variants = pd.DataFrame({'variant_id': [os.path.splitext(n)[0] for n in names], 'source_path': photos})


## Step 2 -- Install the tools and set up the digitiser (same as the main study)


In [ ]:
def sh(cmd, cwd=None, show=True):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, cwd=cwd)
    if show and r.returncode != 0:
        print((r.stdout + r.stderr)[-1500:])
    return r

r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'neurokit2', 'opencv-python-headless', 'yacs', 'scikit-image', 'tqdm', 'torch-tps'],
                   capture_output=True, text=True)
print('pip install:', 'ok' if r.returncode == 0 else 'FAILED\n' + r.stderr[-800:])
import neurokit2 as nk, cv2, torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
if device == 'cpu':
    print('WARNING: no GPU. Right panel -> Settings -> Accelerator -> choose a GPU, then Run All again. On CPU this takes about 2 minutes per image.')



In [ ]:
if os.path.exists(DIG_DIR):
    shutil.rmtree(DIG_DIR)
r = sh(f'git clone https://github.com/Ahus-AIM/Open-ECG-Digitizer.git {DIG_DIR}')
assert os.path.isdir(f'{DIG_DIR}/src'), 'git clone failed. Is Internet switched on? Right panel -> Settings -> Internet -> On.'
sh('apt-get -qq install -y git-lfs', show=False)
sh('git lfs install', cwd=DIG_DIR, show=False)
sh('git lfs pull', cwd=DIG_DIR)
path = f'{DIG_DIR}/src/utils.py'
content = open(path).read()
if 'Stopper = object' not in content:      # patch only once
    content = content.replace("from ray.tune import Stopper",
                              "try:\n    from ray.tune import Stopper\nexcept ImportError:\n    Stopper = object  # ray not needed for inference")
    open(path, 'w').write(content)
wts = {f: os.path.getsize(f'{DIG_DIR}/weights/{f}') for f in os.listdir(f'{DIG_DIR}/weights')} if os.path.isdir(f'{DIG_DIR}/weights') else {}
print('weights:', wts)
assert wts.get('unet_weights_07072025.pt', 0) > 1e6 and wts.get('lead_name_unet_weights_07072025.pt', 0) > 1e6, \
    'The digitiser weight files are missing or only pointers (git-lfs problem): paste the messages above into the chat.'
print('Digitiser ready.')



In [ ]:
def write_digitizer_config(images_path, output_path):
    config_content = f'''MODEL:
  class_path: 'src.model.inference_wrapper.InferenceWrapper'
  KWARGS:
    config:
      SIGNAL_EXTRACTOR:
        class_path: 'src.model.signal_extractor.SignalExtractor'
        KWARGS: {{}}
      PERSPECTIVE_DETECTOR:
        class_path: 'src.model.perspective_detector.PerspectiveDetector'
        KWARGS:
          num_thetas: 250
      DEWARPER:
        class_path: 'src.model.dewarper.Dewarper'
        KWARGS:
          abs_peak_threshold: 0.1
      SEGMENTATION_MODEL:
        class_path: 'src.model.unet.UNet'
        weight_path: './weights/unet_weights_07072025.pt'
        KWARGS:
          num_in_channels: 3
          num_out_channels: 4
          dims: [32, 64, 128, 256, 320, 320, 320, 320]
          depth: 2
      CROPPER:
        class_path: 'src.model.cropper.Cropper'
        KWARGS:
          granularity: 80
          percentiles: [0.02, 0.98]
          alpha: 0.85
      PIXEL_SIZE_FINDER:
        class_path: 'src.model.pixel_size_finder.PixelSizeFinder'
        KWARGS:
          min_number_of_grid_lines: 30
          max_number_of_grid_lines: 70
          lower_grid_line_factor: 0.5
      LAYOUT_IDENTIFIER:
        class_path: 'src.model.lead_identifier.LeadIdentifier'
        config_path: 'src/config/lead_layouts_george-moody-2024.yml'
        unet_config_path: 'src/config/lead_name_unet.yml'
        unet_weight_path: './weights/lead_name_unet_weights_07072025.pt'
        KWARGS:
          debug: false
          device: '{device}'
          possibly_flipped: false
          target_num_samples: 10000
          required_valid_samples: 2
    device: '{device}'
    resample_size: 3000
    rotate_on_resample: true
    enable_timing: false
    apply_dewarping: false
DATA:
  images_path: '{images_path}'
  image_extensions: ['.png', '.jpg', '.jpeg', '.JPG']
  save_mode: 'all'
  layout_should_include_substring: false
  output_path: '{output_path}'
  clear_output_dir_if_exists: true
'''
    with open(f'{DIG_DIR}/batch_config.yml', 'w') as f:
        f.write(config_content)

print("Digitizer config helper ready (same settings as the main study).")



In [ ]:
# Digitiser checkpoint, per photo. A photo is read with OpenCV (so the phone's rotation flag is applied) and saved as PNG for the digitiser; the photo scores use the same pixels.
FS_CANONICAL = 1000.0
OUT_NPZ = f'{WORK}/ownphotos_digitized_12lead'
os.makedirs(OUT_NPZ, exist_ok=True)
RESULTS_PATH = f'{WORK}/results_ownphotos.csv'
LEADS = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

def run_batch(batch):
    imgs, outd = f'{TMP}/images_batch', f'{TMP}/digitized_batch'
    for d in (imgs, outd):
        if os.path.exists(d):
            shutil.rmtree(d)
    os.makedirs(imgs)
    meta = {}
    for _, v in batch.iterrows():
        im = cv2.imread(v['source_path'])
        meta[v['variant_id']] = (None if im is None else im.shape[0], None if im is None else im.shape[1])
        if im is not None:
            cv2.imwrite(f'{imgs}/{v["variant_id"]}.png', im)
    write_digitizer_config(imgs, outd)
    t0 = time.time()
    r = subprocess.run([sys.executable, '-m', 'src.digitize', '--config', f'{DIG_DIR}/batch_config.yml'], cwd=DIG_DIR, capture_output=True, text=True)
    rows = []
    for _, v in batch.iterrows():
        vid = v['variant_id']
        csv_path = f'{outd}/{vid}_timeseries_canonical.csv'
        row = {'variant_id': vid, 'img_h': meta[vid][0], 'img_w': meta[vid][1], 'has_output': False, 'n_II_samples': 0, 'raw_hr_II': None}
        if os.path.exists(csv_path):
            dg = pd.read_csv(csv_path)
            np.savez_compressed(f'{OUT_NPZ}/{vid}.npz', **{c: dg[c].to_numpy(dtype='float32') for c in dg.columns})
            row['has_output'] = True
            if 'II' in dg.columns:
                sig = dg['II'].dropna().to_numpy() / 1000.0
                row['n_II_samples'] = int(len(sig))
                try:
                    pk, _ = nk.ecg_peaks(sig, sampling_rate=FS_CANONICAL)
                    row['raw_hr_II'] = round(float(np.nanmean(nk.ecg_rate(pk, sampling_rate=FS_CANONICAL, desired_length=len(sig)))), 1)
                except Exception:
                    pass
        rows.append(row)
    return pd.DataFrame(rows), time.time() - t0, (r.stdout + r.stderr)

if os.path.exists(RESULTS_PATH):
    results_so_far = pd.read_csv(RESULTS_PATH)
    print(f'Resuming: {len(results_so_far)} photos already done')
else:
    results_so_far = pd.DataFrame()
done = set(results_so_far['variant_id']) if len(results_so_far) else set()
remaining = all_variants[~all_variants['variant_id'].isin(done)].reset_index(drop=True)
print(f'{len(all_variants)} photos in total, {len(remaining)} still to do.')


## Step 3 -- Digitise all photos (checkpointed; smoke test on two photos first)


In [ ]:
smoke, secs, tool_out = run_batch(remaining.iloc[:2].reset_index(drop=True))
print(smoke.to_string())
print(f'smoke test took {secs:.0f} s for {len(smoke)} photos')
if not smoke['has_output'].any():
    print('The digitiser made no output for the first two photos. That can be a real result (screen photos often fail) but may also be a set-up problem. Last lines of its output:')
    print(tool_out[-1800:])
    if 'Traceback' in tool_out or 'Error' in tool_out[-1800:]:
        raise SystemExit('STOPPED on purpose: paste the output above into the chat.')

SUBBATCH_SIZE = 10
SESSION_TIME_BUDGET_S = 5 * 3600
session_start = time.time()
while len(remaining) > 0 and (time.time() - session_start) < SESSION_TIME_BUDGET_S:
    batch = remaining.iloc[:SUBBATCH_SIZE].reset_index(drop=True)
    batch_results, secs, _ = run_batch(batch)
    results_so_far = pd.concat([results_so_far, batch_results], ignore_index=True).drop_duplicates(subset='variant_id', keep='last')
    tmp_path = RESULTS_PATH + '.tmp'
    results_so_far.to_csv(tmp_path, index=False)
    os.replace(tmp_path, RESULTS_PATH)
    n_on_disk = len(pd.read_csv(RESULTS_PATH))
    assert n_on_disk == len(results_so_far), f'SAVE VERIFY FAILED: disk has {n_on_disk}, memory has {len(results_so_far)}'
    remaining = remaining.iloc[SUBBATCH_SIZE:].reset_index(drop=True)
    print(f'Checkpointed {len(batch)} photos ({secs/len(batch):.1f} s/photo). Done {len(results_so_far)}/{len(all_variants)}, {len(remaining)} remaining.')
print('ALL PHOTOS DIGITISED.' if len(remaining) == 0 else f'{len(remaining)} photos remaining: run this cell again.')
print(results_so_far.has_output.value_counts().to_string())


## Step 4 -- The five photo scores of every photo (same code as the competition-capture run)


In [ ]:
import cv2
import numpy as np

def compute_signals(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    blur_score = float(cv2.Laplacian(gray, cv2.CV_64F).var())
    edges = cv2.Canny(gray, 50, 150)
    min_len = int(0.3 * gray.shape[1])
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=100, minLineLength=min_len, maxLineGap=10)
    if lines is None or len(lines) < 3:
        skew_deg = None
    else:
        arr = lines.reshape(-1, 4) if lines.ndim == 3 else lines
        devs = []
        for x1, y1, x2, y2 in arr:
            a = np.degrees(np.arctan2(y2 - y1, x2 - x1)) % 180
            d = a % 90
            devs.append(min(d, 90 - d))
        skew_deg = float(np.median(devs))
    f = np.fft.fftshift(np.fft.fft2(gray.astype(np.float32)))
    mag = np.abs(f)
    h, w = mag.shape; cy, cx = h // 2, w // 2
    yy, xx = np.ogrid[:h, :w]
    r = np.sqrt((yy - cy) ** 2 + (xx - cx) ** 2); r_max = min(cy, cx)
    resolution_score = float(mag[r > 0.5 * r_max].sum() / (mag.sum() + 1e-9))
    contrast_score = float(gray.std())
    col_grad = np.mean(np.abs(np.diff(gray.astype(np.float32), axis=1)), axis=0)
    col_grad = col_grad - col_grad.mean()
    spec = np.abs(np.fft.rfft(col_grad))
    if len(spec) < 4:
        blockiness_score = 0.0
    else:
        k0 = int(round(len(col_grad) / 8))
        band = spec[max(1, k0 - 1):k0 + 2].mean(); bg = np.median(spec[1:])
        blockiness_score = float(band / (bg + 1e-6))
    return dict(blur_score=blur_score, skew_deg=skew_deg, resolution_score=resolution_score,
                contrast_score=contrast_score, blockiness_score=blockiness_score)

SIG_PATH = f'{WORK}/ownphotos_quality_signals.csv'
rows_sig = []
for n, (_, v) in enumerate(all_variants.iterrows(), 1):
    img = cv2.imread(v['source_path'])
    if img is None:
        print('could not read', v['source_path'])
        continue
    rows_sig.append(dict(variant_id=v['variant_id'], **compute_signals(img)))
pd.DataFrame(rows_sig).to_csv(SIG_PATH, index=False)
chk = pd.read_csv(SIG_PATH)
print(f'Verified on disk: {len(chk)} rows in {SIG_PATH} (expected {len(all_variants)}).')
prov = {'written_utc': datetime.datetime.utcnow().isoformat() + 'Z', 'open_ecg_digitizer_commit': sh(f'git -C {DIG_DIR} rev-parse HEAD', show=False).stdout.strip() if os.path.isdir(DIG_DIR) else None,
        'device': device, 'n_photos': int(len(all_variants)), 'weights': sorted(os.listdir(f'{DIG_DIR}/weights')) if os.path.isdir(f'{DIG_DIR}/weights') else None}
json.dump(prov, open(f'{WORK}/ownphotos_provenance.json', 'w'), indent=2)
print(json.dumps(prov, indent=2))
print('ALL OWN PHOTOS COMPLETE. Download: results_ownphotos.csv, ownphotos_quality_signals.csv, ownphotos_provenance.json and the folder ownphotos_digitized_12lead.')
